# Multiple Regression with a Hidden Layer

Open in [Google Colab](https://colab.research.google.com/github/febse/ta2025/blob/main/03-05-ANN-Multiple-Regression.ipynb)

In the [previous chapter](03-04-ANN-Simple-Regression-Hidden-Layer.ipynb) we built a network with one input, a hidden layer with ReLU activation and one output, and derived its gradients. In this chapter the network has $p$ input features. The formulas keep their form, the input $x$ becomes an $n \times p$ matrix $X$ and $W_h$ a $p \times k$ matrix. We

1. write the model and its gradients in matrix notation for any number of inputs (reused in all following chapters),
2. implement the network in NumPy and PyTorch for data with two inputs, and
3. finish with multiple linear regression **without** a hidden layer, which is the special case where the hidden layer is removed.

## Multiple Inputs and a Hidden Layer

We can extend the multiple inputs linear regression model by adding a hidden layer with non-linear activation functionsas we did in the previous chapter for the single input case. In this case the prediction equation for a single observation $x$ (a $1 \times p$ row vector; for $n$ observations replace it with the $n \times p$ matrix $X$) becomes:

:::{.ex-class}
$$
\underbrace{\hat{y}}_{1 \times 1} = \text{ReLU}(\underbrace{x}_{1 \times p} \underbrace{W_h}_{p \times k} + \underbrace{b_h}_{1 \times k}) \underbrace{W_o}_{k \times 1} + \underbrace{b_o}_{1 \times 1}
$$
:::

where $k$ is the number of hidden neurons. 

![Multiple Inputs and a Hidden Layer](./figures/ann/Linear-Regression-Two-Inputs-Hidden-Single-Output.png)

{{< video ./videos/ThreeInputsTwoHiddenLayerOneOutput.mp4 >}}

{{< video ./videos/ThreeInputsTwoHiddenLayerOneOutputVector.mp4 >}}

## Gradients for Multiple Inputs

The derivation of the previous chapter does not depend on the number of inputs. The weight $w_{h,jm}$ connecting input $j$ to hidden neuron $m$ enters the prediction only through the net value $h_{im} = \sum_{j} x_{ij} w_{h,jm} + b_{hm}$, so

:::{.ex-class}
$$
\frac{\partial L}{\partial w_{h,jm}} = -\frac{2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i) \, w_{o,m} \, \text{ReLU}'(h_{im}) \, x_{ij}
$$
:::

For $p = 1$ this is the formula of the previous chapter. The output layer gradients are unchanged, because they depend only on the activations $Z$:

:::{.ex-class}
$$
\underbrace{\frac{\partial L}{\partial W_o}}_{k \times 1} = -\frac{2}{n} \underbrace{Z^\top}_{k \times n} \underbrace{(y - \hat{y})}_{n \times 1}
\qquad
\underbrace{\frac{\partial L}{\partial b_o}}_{1 \times 1} = -\frac{2}{n} \underbrace{\mathbf{1}^\top}_{1 \times n} (y - \hat{y})
$$
:::

With $\Delta = \left( (y - \hat{y}) W_o^\top \right) \odot \text{ReLU}'(H)$ (an $n \times k$ matrix) the hidden layer gradients are

:::{.ex-class}
$$
\underbrace{\frac{\partial L}{\partial W_h}}_{p \times k} = -\frac{2}{n} \underbrace{X^\top}_{p \times n} \Delta
\qquad
\underbrace{\frac{\partial L}{\partial b_h}}_{1 \times k} = -\frac{2}{n} \underbrace{\mathbf{1}^\top}_{1 \times n} \Delta
$$
:::

where $k$ is the number of hidden neurons and $\mathbf{1}$ is a vector of $n$ ones. Gradient descent updates every parameter with $\theta \leftarrow \theta - \eta \frac{\partial L}{\partial \theta}$.

:::{.callout-note}
## Shapes

Always pay close attention to the shapes of the matrices and vectors involved in the computations.

:::

## Implementation

### A hidden layer with two inputs

The NumPy code below handles any number of inputs, because the data matrix enters only through $X W_h$ and $X^\top \Delta$. Here the target depends on two features in a non-linear way, so a linear model cannot fit it well.

In [1]:
import numpy as np
import torch
import torch.nn as nn

rng = np.random.default_rng(1)

n = 300
X_two = rng.uniform(-2, 2, size=(n, 2))
y_two = np.sin(2 * X_two[:, [0]]) + X_two[:, [1]] ** 2 + rng.normal(0, 0.1, size=(n, 1))

def relu(H):
    return np.maximum(0, H)

def relu_prime(H):
    return (H > 0).astype(float)

def train_relu_network(X, y, k=16, learning_rate=0.05, epochs=3000, seed=0):
    rng = np.random.default_rng(seed)
    n, p = X.shape

    W_h = rng.normal(0, np.sqrt(2 / p), size=(p, k))
    b_h = np.zeros((1, k))
    W_o = rng.normal(0, np.sqrt(2 / k), size=(k, 1))
    b_o = np.zeros((1, 1))

    for epoch in range(epochs):
        H = X @ W_h + b_h
        Z = relu(H)
        y_hat = Z @ W_o + b_o

        error = y - y_hat
        dW_o = -(2 / n) * Z.T @ error
        db_o = -(2 / n) * error.sum(keepdims=True)
        Delta = (error @ W_o.T) * relu_prime(H)
        dW_h = -(2 / n) * X.T @ Delta
        db_h = -(2 / n) * Delta.sum(axis=0, keepdims=True)

        W_o -= learning_rate * dW_o
        b_o -= learning_rate * db_o
        W_h -= learning_rate * dW_h
        b_h -= learning_rate * db_h

    return np.mean((y - (relu(X @ W_h + b_h) @ W_o + b_o)) ** 2)

# Compare with the best linear model
design_two = np.c_[np.ones(n), X_two]
linear_fit = design_two @ np.linalg.lstsq(design_two, y_two, rcond=None)[0]

print("MSE of multiple linear regression:", round(float(np.mean((y_two - linear_fit) ** 2)), 4))
print("MSE of the network with a hidden layer (NumPy):", round(float(train_relu_network(X_two, y_two)), 4))

MSE of multiple linear regression: 1.8436
MSE of the network with a hidden layer (NumPy): 0.0302


The same network in PyTorch. As in the previous chapter, PyTorch computes the gradients derived above automatically (`loss.backward()`) and we use the Adam optimizer instead of plain gradient descent.

In [2]:
torch.manual_seed(0)

X_two_t = torch.from_numpy(X_two).float()
y_two_t = torch.from_numpy(y_two).float()

model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(3000):
    loss = criterion(model(X_two_t), y_two_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("MSE of the network with a hidden layer (PyTorch):", round(loss.item(), 4))

MSE of the network with a hidden layer (PyTorch): 0.0175


## Multiple Linear Regression Without a Hidden Layer

If we remove the hidden layer, the output neuron is connected directly to the inputs and uses the identity activation. This is the multiple linear regression model, the special case of the network above with $Z = X$, $W_o = W$ and $\text{ReLU}$ replaced by the identity.

With two or more predictor variables (features) the multiple linear regression model predicts an outcome variable as a linear function of the predictor variables. For a single observation with two predictors:

$$
\hat{y} = w_1 x_1 + w_2 x_2 + b
$$

It is convenient to write this equation in vectorized form for all $n$ observations at once.

Let $X$ be an $n \times p$ matrix of input features (here $p = 2$) and $W$ a $p \times 1$ vector of weights:

$$
X = \begin{bmatrix}
x_{11} & x_{12} \\
x_{21} & x_{22} \\
\vdots & \vdots \\
x_{n1} & x_{n2}
\end{bmatrix}
\quad 
W = \begin{bmatrix}
w_{1} \\
w_{2} \\
\vdots \\
w_{p}
\end{bmatrix}
$$

Then the equation for the multiple linear regression model is:

:::{.ex-class}
$$
\hat{y} = X W + b
$$
:::

where $\hat{y}$ is the $n \times 1$ vector of predictions.

![Multiple Regression as a Neural Network](./figures/ann/Linear-Regression-Two-Inputs-Single-Output.png)

### Gradients of the multiple regression model

The loss is the same mean squared error as before,

$$
L = \text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
\qquad\text{with}\qquad
\hat{y}_i = \sum_{j=1}^{p} x_{ij} w_j + b
$$

By the chain rule, $\frac{\partial L}{\partial \hat{y}_i} = -\frac{2}{n}(y_i - \hat{y}_i)$, and $\frac{\partial \hat{y}_i}{\partial w_j} = x_{ij}$, $\frac{\partial \hat{y}_i}{\partial b} = 1$. Therefore

:::{.ex-class}
$$
\frac{\partial L}{\partial w_j} = -\frac{2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i) x_{ij}
\qquad
\frac{\partial L}{\partial b} = -\frac{2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)
$$
:::

The first expression is a scaled dot product of the error vector $(y - \hat{y})$ and the $j$-th column of $X$. Stacking all $p$ partial derivatives gives the vectorized gradients

:::{.ex-class}
$$
\underbrace{\frac{\partial L}{\partial W}}_{p \times 1} = -\frac{2}{n} \underbrace{X^\top}_{p \times n} \underbrace{(y - \hat{y})}_{n \times 1}
\qquad
\underbrace{\frac{\partial L}{\partial b}}_{1 \times 1} = -\frac{2}{n} \underbrace{\mathbf{1}^\top}_{1 \times n} \underbrace{(y - \hat{y})}_{n \times 1}
$$
:::

where $\mathbf{1}$ is a vector of $n$ ones. For $p = 1$ these are the formulas for the simple regression model. Gradient descent updates the parameters with $W \leftarrow W - \eta \frac{\partial L}{\partial W}$ and $b \leftarrow b - \eta \frac{\partial L}{\partial b}$. Setting the gradients to zero gives the normal equations of ordinary least squares, so (for a small enough learning rate) gradient descent converges to the OLS solution. This is checked numerically below.

### Gradient descent versus OLS

We generate data from the model $y = 1 + 2 x_1 - 3 x_2 + \varepsilon$, fit it with gradient descent using the gradients derived above, and compare the result with the ordinary least squares solution.

In [3]:
rng = np.random.default_rng(42)

n = 200
p = 2
X_lin = rng.normal(size=(n, p))                                                                   # n x p
y_lin = 1 + 2 * X_lin[:, [0]] - 3 * X_lin[:, [1]] + rng.normal(0, 0.5, size=(n, 1))               # n x 1

# Initialize and train with plain full-batch gradient descent
W = np.zeros((p, 1))
b = np.zeros((1, 1))
learning_rate = 0.1

for epoch in range(500):
    y_hat = X_lin @ W + b
    error = y_lin - y_hat

    dW = -(2 / n) * X_lin.T @ error                  # p x 1
    db = -(2 / n) * error.sum(keepdims=True)         # 1 x 1

    W -= learning_rate * dW
    b -= learning_rate * db

# Ordinary least squares (add a column of ones for the intercept)
design = np.c_[np.ones(n), X_lin]
ols_coefficients = np.linalg.lstsq(design, y_lin, rcond=None)[0].ravel()

print("Gradient descent: b =", b.ravel().round(4), " W =", W.ravel().round(4))
print("OLS:              b =", ols_coefficients[:1].round(4), " W =", ols_coefficients[1:].round(4))

Gradient descent: b = [0.9653]  W = [ 1.9911 -3.0267]
OLS:              b = [0.9653]  W = [ 1.9911 -3.0267]
